# Importing Libraries


In [22]:
import pandas as pd
from pathlib import Path
import sys
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import joblib
from sklearn.model_selection import GridSearchCV
import numpy as np

In [2]:
current_dir = Path.cwd().parent
sys.path.append(str(current_dir))
from src.config import RAW_DATA_DIR, PROCESSED_DATA_DIR, MODEL_DIR

In [3]:
OUTPUT_FILE_X = PROCESSED_DATA_DIR / "housing_prepared_x.pkl"
OUTPUT_FILE_Y = PROCESSED_DATA_DIR / "housing_labels.pkl"

In [4]:
forest_pkl = MODEL_DIR / "random_forest_v1.pkl"

## Load the data


### X


In [6]:
housing_prepared = joblib.load(OUTPUT_FILE_X)
print("Data loaded successfully.")

Data loaded successfully.


### Y


In [7]:
housing_labels = joblib.load(OUTPUT_FILE_Y)
print("Data loaded successfully.")

Data loaded successfully.


## Get the model


In [8]:
experiment_forest = joblib.load(forest_pkl)

In [9]:
old_forest_model = experiment_forest["model_object"]

In [10]:
print("Old Random Forest model loaded:")
print(old_forest_model)

Old Random Forest model loaded:
RandomForestRegressor(random_state=42)


In [11]:
print("\nOld mean RMSE:")
print(experiment_forest["mean_rmse"])


Old mean RMSE:
50435.58092066179


# fine tune the model


## GridSearch


### PARAM_GRID 1


In [17]:
# PARAMETER GRID DEFINITION

param_grid = [
    {"n_estimators": [3, 10, 30], "max_features": [2, 4, 6, 8]},
    {"bootstrap": [False], "n_estimators": [3, 10], "max_features": [2, 3, 4]},
]

In [18]:
grid_search = GridSearchCV(
    old_forest_model,
    param_grid,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)

In [19]:
grid_search.fit(housing_prepared, housing_labels)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'max_features': [2, 4, ...], 'n_estimators': [3, 10, ...]}, {'bootstrap': [False], 'max_features': [2, 3, ...], 'n_estimators': [3, 10]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` w

In [20]:
print("\nBest parameters:")
print(grid_search.best_params_)


Best parameters:
{'max_features': 8, 'n_estimators': 30}


In [21]:
print("\nBest estimator:")
print(grid_search.best_estimator_)


Best estimator:
RandomForestRegressor(max_features=8, n_estimators=30, random_state=42)


In [23]:
best_rmse = np.sqrt(-grid_search.best_score_)

print("\nBest CV RMSE:")
print(best_rmse)


Best CV RMSE:
49898.98913455217


In [24]:
cv_results = grid_search.cv_results_

print("\nAll Grid Search Results:")
for mean_score, params in zip(cv_results["mean_test_score"], cv_results["params"]):
    rmse = np.sqrt(-mean_score)
    print(rmse, params)


All Grid Search Results:
63895.161577951665 {'max_features': 2, 'n_estimators': 3}
54916.32386349543 {'max_features': 2, 'n_estimators': 10}
52885.86715332332 {'max_features': 2, 'n_estimators': 30}
60075.3680329983 {'max_features': 4, 'n_estimators': 3}
52495.01284985185 {'max_features': 4, 'n_estimators': 10}
50187.24324926565 {'max_features': 4, 'n_estimators': 30}
58064.73529982314 {'max_features': 6, 'n_estimators': 3}
51519.32062366315 {'max_features': 6, 'n_estimators': 10}
49969.80441627874 {'max_features': 6, 'n_estimators': 30}
58895.824998155826 {'max_features': 8, 'n_estimators': 3}
52459.79624724529 {'max_features': 8, 'n_estimators': 10}
49898.98913455217 {'max_features': 8, 'n_estimators': 30}
62381.765106921855 {'bootstrap': False, 'max_features': 2, 'n_estimators': 3}
54476.57050944266 {'bootstrap': False, 'max_features': 2, 'n_estimators': 10}
59974.60028085155 {'bootstrap': False, 'max_features': 3, 'n_estimators': 3}
52754.5632813202 {'bootstrap': False, 'max_featu

In [28]:
results_df = pd.DataFrame(
    {
        "params": cv_results["params"],
        "train_rmse": np.sqrt(-cv_results["mean_train_score"]),
        "validation_rmse": np.sqrt(-cv_results["mean_test_score"]),
        "gap": np.sqrt(-cv_results["mean_test_score"])
        - np.sqrt(-cv_results["mean_train_score"]),
        "rank": cv_results["rank_test_score"],
    }
)

results_df = results_df.sort_values("rank")

print(results_df)

                                               params    train_rmse  \
11            {'max_features': 8, 'n_estimators': 30}  19521.531601   
8             {'max_features': 6, 'n_estimators': 30}  19562.671288   
5             {'max_features': 4, 'n_estimators': 30}  19641.467005   
17  {'bootstrap': False, 'max_features': 4, 'n_est...     -0.000000   
7             {'max_features': 6, 'n_estimators': 10}  22457.055965   
10            {'max_features': 8, 'n_estimators': 10}  22579.696207   
4             {'max_features': 4, 'n_estimators': 10}  22741.879229   
15  {'bootstrap': False, 'max_features': 3, 'n_est...     -0.000000   
2             {'max_features': 2, 'n_estimators': 30}  20940.410082   
13  {'bootstrap': False, 'max_features': 2, 'n_est...     13.784984   
1             {'max_features': 2, 'n_estimators': 10}  24349.319998   
16  {'bootstrap': False, 'max_features': 4, 'n_est...     -0.000000   
6              {'max_features': 6, 'n_estimators': 3}  30202.645036   
9     

### PARAM GRID 2


In [33]:
# PARAMETER GRID DEFINITION
# Reduce model complexity - overfitting solution
# expand the grid - Best value is at the maximum
param_grid_2 = [
    {
        "n_estimators": [50, 100, 200],
        "max_features": [8, 10, 12],
        "max_depth": [None, 20, 40],  # trees cannot grow too deep
        "min_samples_leaf": [1, 2, 4],  # each leaf must contain more samples
    },
]

In [34]:
grid_search_2 = GridSearchCV(
    old_forest_model,
    param_grid_2,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)

In [35]:
grid_search_2.fit(housing_prepared, housing_labels)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'max_depth': [None, 20, ...], 'max_features': [8, 10, ...], 'min_samples_leaf': [1, 2, ...], 'n_estimators': [50, 100, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be 

In [36]:
print("\nBest parameters:")
print(grid_search_2.best_params_)


Best parameters:
{'max_depth': 40, 'max_features': 8, 'min_samples_leaf': 1, 'n_estimators': 200}


In [37]:
print("\nBest estimator:")
print(grid_search_2.best_estimator_)


Best estimator:
RandomForestRegressor(max_depth=40, max_features=8, n_estimators=200,
                      random_state=42)


In [38]:
best_rmse_2 = np.sqrt(-grid_search_2.best_score_)

print("\nBest CV RMSE:")
print(best_rmse_2)


Best CV RMSE:
49183.39280229622


In [44]:
cv_results_2 = grid_search_2.cv_results_

print("\nAll Grid Search Results:")
for mean_score, params in zip(cv_results_2["mean_test_score"], cv_results_2["params"]):
    rmse = np.sqrt(-mean_score)
    print(rmse, params)


All Grid Search Results:
49547.0229085019 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 1, 'n_estimators': 50}
49339.806213071206 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 1, 'n_estimators': 100}
49186.59493765748 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 1, 'n_estimators': 200}
49748.93754833343 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 2, 'n_estimators': 50}
49570.232019138326 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 2, 'n_estimators': 100}
49403.894306152746 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 2, 'n_estimators': 200}
49953.954167576 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 4, 'n_estimators': 50}
49813.507847557084 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 4, 'n_estimators': 100}
49710.33232480466 {'max_depth': None, 'max_features': 8, 'min_samples_leaf': 4, 'n_estimators': 200}
49774.245557689195 {'max_depth': None, 'max_features': 10, 'min_sam

In [45]:
results_2df = pd.DataFrame(
    {
        "params": cv_results_2["params"],
        "train_rmse": np.sqrt(-cv_results_2["mean_train_score"]),
        "validation_rmse": np.sqrt(-cv_results_2["mean_test_score"]),
        "gap": np.sqrt(-cv_results_2["mean_test_score"])
        - np.sqrt(-cv_results_2["mean_train_score"]),
        "rank": cv_results_2["rank_test_score"],
    }
)

results_2df = results_2df.sort_values("rank")

In [52]:
print(results_2df[results_2df["rank"] == 1])

                                               params    train_rmse  \
56  {'max_depth': 40, 'max_features': 8, 'min_samp...  18334.562817   

    validation_rmse           gap  rank  
56     49183.392802  30848.829985     1  


### PARAM GRID 3


In [55]:
param_grid_3 = [
    {
        "n_estimators": [200, 300, 500],
        "max_features": [4, 6, 8],
        "max_depth": [30, 40, 50],
        "min_samples_leaf": [1, 2],
    }
]

In [56]:
grid_search_3 = GridSearchCV(
    old_forest_model,
    param_grid_3,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)

In [57]:
grid_search_3.fit(housing_prepared, housing_labels)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'max_depth': [30, 40, ...], 'max_features': [4, 6, ...], 'min_samples_leaf': [1, 2], 'n_estimators': [200, 300, ...]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be availab

In [58]:
print("\nBest parameters:")
print(grid_search_3.best_params_)


Best parameters:
{'max_depth': 30, 'max_features': 6, 'min_samples_leaf': 1, 'n_estimators': 500}


In [59]:
print("\nBest estimator:")
print(grid_search_3.best_estimator_)


Best estimator:
RandomForestRegressor(max_depth=30, max_features=6, n_estimators=500,
                      random_state=42)


In [60]:
best_rmse_3 = np.sqrt(-grid_search_3.best_score_)

print("\nBest CV RMSE:")
print(best_rmse_3)


Best CV RMSE:
48997.19657232936


In [61]:
cv_results_3 = grid_search_3.cv_results_

print("\nAll Grid Search Results:")
for mean_score, params in zip(cv_results_3["mean_test_score"], cv_results_3["params"]):
    rmse = np.sqrt(-mean_score)
    print(rmse, params)


All Grid Search Results:
49452.153042077276 {'max_depth': 30, 'max_features': 4, 'min_samples_leaf': 1, 'n_estimators': 200}
49445.15346348593 {'max_depth': 30, 'max_features': 4, 'min_samples_leaf': 1, 'n_estimators': 300}
49384.701505304176 {'max_depth': 30, 'max_features': 4, 'min_samples_leaf': 1, 'n_estimators': 500}
49741.72891653265 {'max_depth': 30, 'max_features': 4, 'min_samples_leaf': 2, 'n_estimators': 200}
49715.530280286024 {'max_depth': 30, 'max_features': 4, 'min_samples_leaf': 2, 'n_estimators': 300}
49617.80113251823 {'max_depth': 30, 'max_features': 4, 'min_samples_leaf': 2, 'n_estimators': 500}
49176.0410729676 {'max_depth': 30, 'max_features': 6, 'min_samples_leaf': 1, 'n_estimators': 200}
49113.37610055177 {'max_depth': 30, 'max_features': 6, 'min_samples_leaf': 1, 'n_estimators': 300}
48997.19657232936 {'max_depth': 30, 'max_features': 6, 'min_samples_leaf': 1, 'n_estimators': 500}
49333.18561037395 {'max_depth': 30, 'max_features': 6, 'min_samples_leaf': 2, 'n_

In [62]:
results_3df = pd.DataFrame(
    {
        "params": cv_results_3["params"],
        "train_rmse": np.sqrt(-cv_results_3["mean_train_score"]),
        "validation_rmse": np.sqrt(-cv_results_3["mean_test_score"]),
        "gap": np.sqrt(-cv_results_3["mean_test_score"])
        - np.sqrt(-cv_results_3["mean_train_score"]),
        "rank": cv_results_3["rank_test_score"],
    }
)

results_3df = results_3df.sort_values("rank")

In [63]:
print(results_3df[results_3df["rank"] == 1])

                                              params    train_rmse  \
8  {'max_depth': 30, 'max_features': 6, 'min_samp...  18109.880445   

   validation_rmse           gap  rank  
8     48997.196572  30887.316127     1  


### PARAM GRID 4


In [64]:
param_grid_reduce_overfit = [
    {
        "n_estimators": [200, 300],
        "max_features": [4, 6, 8],
        "max_depth": [20, 30, 40],
        "min_samples_leaf": [2, 4, 6],
    }
]

In [65]:
grid_search_overfit = GridSearchCV(
    old_forest_model,
    param_grid_reduce_overfit,
    cv=5,
    scoring="neg_mean_squared_error",
    return_train_score=True,
)

In [66]:
grid_search_overfit.fit(housing_prepared, housing_labels)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","[{'max_depth': [20, 30, ...], 'max_features': [4, 6, ...], 'min_samples_leaf': [2, 4, ...], 'n_estimators': [200, 300]}]"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be availab

In [67]:
print("\nBest parameters:")
print(grid_search_overfit.best_params_)


Best parameters:
{'max_depth': 20, 'max_features': 6, 'min_samples_leaf': 2, 'n_estimators': 300}


In [68]:
print("\nBest estimator:")
print(grid_search_overfit.best_estimator_)


Best estimator:
RandomForestRegressor(max_depth=20, max_features=6, min_samples_leaf=2,
                      n_estimators=300, random_state=42)


In [69]:
best_rmse_overfit = np.sqrt(-grid_search_overfit.best_score_)

print("\nBest CV RMSE:")
print(best_rmse_overfit)


Best CV RMSE:
49231.83620652532


In [72]:
cv_results_overfit = grid_search_overfit.cv_results_

print("\nAll Grid Search Results:")
for mean_score, params in zip(
    cv_results_overfit["mean_test_score"], cv_results_overfit["params"]
):
    rmse = np.sqrt(-mean_score)
    print(rmse, params)


All Grid Search Results:
49767.64764924533 {'max_depth': 20, 'max_features': 4, 'min_samples_leaf': 2, 'n_estimators': 200}
49756.35356335055 {'max_depth': 20, 'max_features': 4, 'min_samples_leaf': 2, 'n_estimators': 300}
50499.473941869706 {'max_depth': 20, 'max_features': 4, 'min_samples_leaf': 4, 'n_estimators': 200}
50475.903529908464 {'max_depth': 20, 'max_features': 4, 'min_samples_leaf': 4, 'n_estimators': 300}
51166.88692269373 {'max_depth': 20, 'max_features': 4, 'min_samples_leaf': 6, 'n_estimators': 200}
51130.9629298399 {'max_depth': 20, 'max_features': 4, 'min_samples_leaf': 6, 'n_estimators': 300}
49313.259918586155 {'max_depth': 20, 'max_features': 6, 'min_samples_leaf': 2, 'n_estimators': 200}
49231.83620652532 {'max_depth': 20, 'max_features': 6, 'min_samples_leaf': 2, 'n_estimators': 300}
49821.26055069008 {'max_depth': 20, 'max_features': 6, 'min_samples_leaf': 4, 'n_estimators': 200}
49709.614092813805 {'max_depth': 20, 'max_features': 6, 'min_samples_leaf': 4, 'n

In [73]:
results_overfit_df = pd.DataFrame(
    {
        "params": cv_results_overfit["params"],
        "train_rmse": np.sqrt(-cv_results_overfit["mean_train_score"]),
        "validation_rmse": np.sqrt(-cv_results_overfit["mean_test_score"]),
        "gap": np.sqrt(-cv_results_overfit["mean_test_score"])
        - np.sqrt(-cv_results_overfit["mean_train_score"]),
        "rank": cv_results_overfit["rank_test_score"],
    }
)

results_overfit_df = results_overfit_df.sort_values("rank")

In [74]:
print(results_overfit_df[results_overfit_df["rank"] == 1])

                                              params    train_rmse  \
7  {'max_depth': 20, 'max_features': 6, 'min_samp...  24870.728611   

   validation_rmse           gap  rank  
7     49231.836207  24361.107595     1  
